# RipeSense — fruit detection and spectral-data audit
Run on **Google Colab**, Python 3, CPU for the default path; choose **T4 GPU** for optional fine-tuning.
No local data or hardware is required. Our three demo apples (including the rotten apple) are not training data.

1. Install → mount Drive → run public-image detection and export RGB crops.
2. Optionally enable COCO fine-tuning and compare both detectors on the same validation subset.
3. Run the Mendeley inventory and sample-image inspection. **Spectral accuracy is withheld unless specimen identity, stage and wavelength mappings are verified.**
4. Read the demo handoff; artifacts are written to a timestamped Drive run directory.

Reference: Sujal's `src/RipeSense_Train.ipynb` uses dataset `y5h6v8w6ms`, version 2 and a `ripeness` filename filter. This notebook does not reuse its image-level splitting or infer labels from days.


In [ ]:
%pip -q install ultralytics==8.3.235 remotezip==0.12.3 scikit-learn pandas pillow matplotlib pyyaml joblib


In [ ]:
from pathlib import Path
import os, json, random, hashlib, shutil, re, io, zipfile, sys, subprocess
from datetime import datetime, timezone
from collections import Counter, defaultdict
import requests, numpy as np, pandas as pd
from PIL import Image, ImageOps, ImageDraw
from IPython.display import display
import matplotlib.pyplot as plt
from google.colab import drive

drive.mount('/content/drive')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
OUT = Path('/content/drive/MyDrive/RipeSense_Detection_Spectral') / RUN_ID
WORK = Path('/content/ripesense_detection_spectral')
OUT.mkdir(parents=True, exist_ok=True); WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
SEED = 42
random.seed(SEED); np.random.seed(SEED)
RUN_FINETUNE = False  # set True and rerun optional cells; baseline remains unchanged
TRAIN_PER_CLASS, VAL_PER_CLASS, EPOCHS = 120, 40, 5
S = requests.Session()
S.headers.update({'User-Agent': 'RipeSense-research/1.0'})

def save_json(name, value):
    (OUT / name).write_text(json.dumps(value, indent=2, default=str))

def download(url, dest, max_bytes=2_000_000_000, expected_size=None):
    dest = Path(dest); dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size and (expected_size is None or dest.stat().st_size == expected_size):
        return dest
    tmp = dest.with_suffix(dest.suffix + '.part')
    try:
        with S.get(url, stream=True, timeout=(30, 180)) as r:
            r.raise_for_status()
            if int(r.headers.get('Content-Length', 0)) > max_bytes:
                raise ValueError('Download exceeds configured size limit')
            total = 0
            with tmp.open('wb') as f:
                for chunk in r.iter_content(1024 * 1024):
                    total += len(chunk)
                    if total > max_bytes: raise ValueError('Download exceeds size limit')
                    f.write(chunk)
        if expected_size is not None and total != expected_size:
            raise ValueError(f'Incomplete download: {total} != {expected_size}')
        tmp.replace(dest)
    finally:
        tmp.unlink(missing_ok=True)
    return dest

save_json('settings.json', dict(seed=SEED, run_finetune=RUN_FINETUNE,
          train_per_class=TRAIN_PER_CLASS, val_per_class=VAL_PER_CLASS, epochs=EPOCHS))
(OUT / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))
print('Results and models:', OUT, '\nTemporary data:', WORK)


## 1 — Pretrained detector: run today
Use the small COCO-pretrained **YOLO11n**. Supported fruit categories are apple, banana and orange; detection is not a ripeness prediction. Other fruit species may be missed or misidentified.
The public COCO fruit photograph (val2017 image 189078) is fetched directly; every detected target gets a numbered box and an unresized RGB crop. No classifier weights from Sujal are required for this step.


In [ ]:
from ultralytics import YOLO
BASELINE_WEIGHTS = WORK / 'yolo11n.pt'
baseline = YOLO(str(BASELINE_WEIGHTS))
shutil.copy2(BASELINE_WEIGHTS, OUT / 'detector_pretrained_yolo11n.pt')
FRUIT_NAMES = {'apple', 'banana', 'orange'}
FRUIT_IDS = sorted(k for k, v in baseline.names.items() if v in FRUIT_NAMES)
assert {baseline.names[k] for k in FRUIT_IDS} == FRUIT_NAMES
print('COCO model class IDs:', {k: baseline.names[k] for k in FRUIT_IDS})
EXAMPLE_URL = 'https://s3.amazonaws.com/images.cocodataset.org/val2017/000000189078.jpg'
example = download(EXAMPLE_URL, WORK / 'coco_000000189078.jpg', max_bytes=20_000_000)
save_json('demo_source.json', {'url': EXAMPLE_URL, 'coco_image_id': 189078, 'original_url': 'http://farm5.staticflickr.com/4026/4430551771_58c190b055_z.jpg', 'license': 'https://creativecommons.org/licenses/by/2.0/', 'sha256': hashlib.sha256(example.read_bytes()).hexdigest()})

def detect_and_export(image_path, model=baseline, tag='public_baseline', conf=0.25):
    """Also accepts a future phone/gantry RGB file. Box IDs are per image, not tracked identities."""
    dest = OUT / tag; dest.mkdir(parents=True, exist_ok=True)
    rgb = ImageOps.exif_transpose(Image.open(image_path)).convert('RGB')
    ids = [k for k, v in model.names.items() if v in FRUIT_NAMES]
    result = model.predict(rgb, classes=ids, conf=conf, imgsz=640, verbose=False)[0]
    canvas = rgb.copy(); draw = ImageDraw.Draw(canvas)
    records = []
    boxes = sorted(result.boxes.data.cpu().tolist(), key=lambda b: (b[0], b[1]))
    for n, (x1, y1, x2, y2, score, cls) in enumerate(boxes, 1):
        x1, y1 = max(0, int(np.floor(x1))), max(0, int(np.floor(y1)))
        x2, y2 = min(rgb.width, int(np.ceil(x2))), min(rgb.height, int(np.ceil(y2)))
        if x2 <= x1 or y2 <= y1: continue
        label = model.names[int(cls)]
        crop = f'{n:03d}_{label}.png'
        rgb.crop((x1,y1,x2,y2)).save(dest / crop)
        draw.rectangle((x1,y1,x2,y2), outline='red', width=3)
        draw.text((x1, max(0,y1-14)), f'{n}: {label} {score:.2f}', fill='white', stroke_width=2, stroke_fill='black')
        records.append(dict(detection_id=n, fruit=label, confidence=score,
                            x1=x1,y1=y1,x2=x2,y2=y2,crop=crop,source=str(image_path)))
    canvas.save(dest / 'numbered_boxes.jpg')
    pd.DataFrame(records, columns=['detection_id','fruit','confidence','x1','y1','x2','y2','crop','source']).to_csv(dest / 'detections.csv', index=False)
    display(canvas); display(pd.DataFrame(records))
    if not records: print('No target fruits detected. No crops fabricated; try another image or inspect confidence.')
    return records

public_detections = detect_and_export(example)


**Sujal handoff:** `public_baseline/detections.csv` links each numbered box to its original-resolution RGB PNG. Apply the exact resize, normalization and class order exported by Sujal's trained model; do not interpret detector confidence as ripeness confidence. The original notebook exports ONNX plus a labels JSON, but those trained files are not present here. Keep `over-ripe` separate from `spoiled`; translate spelling such as `overripe` only, never merge these meanings.

## Optional — small COCO fine-tuning subset
Downloads the official **annotations only** (~252 MB ZIP) and at most 360 train + 120 validation images, not the full COCO image archives. Uses `train2017` and `val2017` separately, retains every target bounding box in selected images, and excludes images with target crowd annotations (YOLO text boxes cannot express crowd-ignore regions). Caps are per class, with overlapping images deduplicated.

Keep the original 80-class head and original class indices to allow an identical evaluation configuration for baseline and fine-tuned weights. Only target fruit labels are used. This is a fruit-only experiment; performance on other COCO classes is not assessed. COCO pretraining already used train2017; this comparison is not evidence of unseen-domain performance. Validation is used for checkpoint selection, not a final independent test.


In [ ]:
def coco_box_to_yolo(ann, width, height, class_id):
    x,y,w,h = ann['bbox']
    x1,y1 = max(0,x),max(0,y)
    x2,y2 = min(width,x+w),min(height,y+h)
    if x2 <= x1 or y2 <= y1: raise ValueError('Degenerate COCO box')
    return [class_id,(x1+x2)/(2*width),(y1+y2)/(2*height),(x2-x1)/width,(y2-y1)/height]

def prepare_coco_subset():
    import yaml
    annzip = download('https://s3.amazonaws.com/images.cocodataset.org/annotations/annotations_trainval2017.zip',
                      WORK / 'annotations_trainval2017.zip', max_bytes=350_000_000)
    root = WORK / f'coco_fruit_{RUN_ID}'; manifests=[]; split_ids={}
    with zipfile.ZipFile(annzip) as z:
        for split, cap in [('train',TRAIN_PER_CLASS), ('val',VAL_PER_CLASS)]:
            data = json.loads(z.read(f'annotations/instances_{split}2017.json'))
            cat_to_model = {c['id']: next(k for k,v in baseline.names.items() if v == c['name'])
                            for c in data['categories'] if c['name'] in FRUIT_NAMES}
            grouped=defaultdict(list); crowd=set()
            for a in data['annotations']:
                if a['category_id'] in cat_to_model:
                    grouped[a['image_id']].append(a)
                    if a.get('iscrowd'): crowd.add(a['image_id'])
            selected=set(); rng=random.Random(SEED)
            for category in sorted(cat_to_model):
                candidates=sorted(i for i,aa in grouped.items() if i not in crowd and any(a['category_id']==category for a in aa))
                rng.shuffle(candidates); selected.update(candidates[:cap])
            split_ids[split]=selected
            image_dir=root/'images'/split; label_dir=root/'labels'/split
            image_dir.mkdir(parents=True); label_dir.mkdir(parents=True)
            for im in data['images']:
                if im['id'] not in selected: continue
                filename=im['file_name']
                download(f'https://s3.amazonaws.com/images.cocodataset.org/{split}2017/{filename}',image_dir/filename,max_bytes=25_000_000)
                with Image.open(image_dir/filename) as decoded:
                    assert decoded.size == (im['width'],im['height'])
                labels=[coco_box_to_yolo(a,im['width'],im['height'],cat_to_model[a['category_id']]) for a in grouped[im['id']]]
                (label_dir / (Path(filename).stem+'.txt')).write_text('\n'.join(' '.join(map(str,row)) for row in labels)+'\n')
                manifests.append(dict(split=split,image_id=im['id'],filename=filename,boxes=len(labels),
                                      annotation_ids=[a['id'] for a in grouped[im['id']]],
                                      sha256=hashlib.sha256((image_dir/filename).read_bytes()).hexdigest(),
                                      source=im.get('coco_url'),license_id=im.get('license')))
            save_json(f'coco_{split}_licenses.json',data['licenses'])
    assert not (split_ids['train'] & split_ids['val'])
    m=pd.DataFrame(manifests)
    assert not (set(m[m.split=='train'].sha256) & set(m[m.split=='val'].sha256)), 'Duplicate images across splits'
    m.to_csv(OUT/'coco_subset_manifest.csv',index=False)
    config=dict(path=str(root),train='images/train',val='images/val',names=baseline.names)
    yamlpath=root/'fruit.yaml'; yamlpath.write_text(yaml.safe_dump(config))
    shutil.copy2(yamlpath,OUT/'coco_fruit.yaml')
    display(m.groupby('split').agg(images=('image_id','count'),boxes=('boxes','sum')))
    return yamlpath

if RUN_FINETUNE:
    coco_yaml = prepare_coco_subset()
else:
    print('Optional download and fine-tuning skipped. Set RUN_FINETUNE=True to enable.')


In [ ]:
if RUN_FINETUNE:
    import torch
    device = 0 if torch.cuda.is_available() else 'cpu'
    eval_args=dict(data=str(coco_yaml),classes=FRUIT_IDS,imgsz=640,batch=8,device=device,plots=True)
    before=YOLO(str(OUT/'detector_pretrained_yolo11n.pt')).val(**eval_args,project=str(OUT/'evaluation'),name='baseline')
    finetuned=YOLO(str(OUT/'detector_pretrained_yolo11n.pt'))
    finetuned.train(data=str(coco_yaml),classes=FRUIT_IDS,epochs=EPOCHS,imgsz=640,batch=8,
                    device=device,seed=SEED,workers=2,project=str(OUT/'training'),name='fruit_finetune')
    best=Path(finetuned.trainer.best)
    shutil.copy2(best,OUT/'detector_finetuned_best.pt')
    finetuned=YOLO(str(OUT/'detector_finetuned_best.pt'))
    after=finetuned.val(**eval_args,project=str(OUT/'evaluation'),name='finetuned')
    comparison=pd.DataFrame([dict(model=name,**metrics.results_dict) for name,metrics in [('pretrained',before),('finetuned',after)]])
    comparison.to_csv(OUT/'detector_comparison.csv',index=False); display(comparison)
    detect_and_export(example,finetuned,'public_finetuned')


## 2 — Inspect Mendeley ripeness files before defining spectral features
[Gaikwad & Kothari, Mendeley Data v2](https://doi.org/10.17632/y5h6v8w6ms.2), CC BY 4.0. Only `Ripeness Grading.zip` is accessed. We inspect its central directory and a nine-image observation using HTTP range reads. If the server stops supporting ranges, the explicit fallback downloads the 1.56 GB ripeness ZIP to temporary Colab storage. No other dataset portions are downloaded.

**Verified during development (2026-11-07):** 3,231 JPEGs, 18 day folders, five variety folders and four apple-number folders per variety. Day 9 has 171 images; other days have 180. Example: `Ripeness Grading/day 1/kinnaur/APPLE 1/apple1_1.jpg`. The nine downloaded sample JPEGs are **17,280 × 1,080 concatenated strips**, each visually containing nine 1,920-pixel-wide panels (one dark-looking panel and eight illuminated panels). This is layout evidence, not a verified wavelength/reference key. There are no metadata tables or scripts inside this ZIP. The publisher lists under-ripe, ripe, over-ripe and eight illumination bands, but the archive has filename suffixes 1–9, no wavelength values and no stage folders. The numbers must not be treated as nanometres or assigned to AS7341 channels.


In [ ]:
from remotezip import RemoteZip
MD_API='https://data.mendeley.com/public-api/datasets/y5h6v8w6ms'
r=S.get(MD_API+'/files',params={'folder_id':'root','version':2,'$start':0,'$limit':1000},
        headers={'Accept':'application/vnd.mendeley-public-dataset.1+json'},timeout=60)
r.raise_for_status(); md_files=r.json()
ripeness=[f for f in md_files if f['filename'].lower()=='ripeness grading.zip']
assert len(ripeness)==1, 'Ripeness archive changed: inspect metadata before continuing'
meta=ripeness[0]; details=meta['content_details']; archive_url=details['download_url']
save_json('mendeley_file_metadata.json',meta)
LOCAL_ARCHIVE=None
try:
    with RemoteZip(archive_url,timeout=120) as z:
        members=z.namelist()
except Exception as e:
    print('Range inventory unavailable:',repr(e),'— downloading ripeness ZIP only.')
    LOCAL_ARCHIVE=download(archive_url,WORK/'Ripeness Grading.zip',expected_size=details['size'])
    assert hashlib.file_digest(LOCAL_ARCHIVE.open('rb'),'sha256').hexdigest()==details['sha256_hash']
    with zipfile.ZipFile(LOCAL_ARCHIVE) as z: members=z.namelist()
save_json('mendeley_archive_members.json',members)
files_in_zip=[n for n in members if not n.endswith('/')]
print('File types:',Counter(Path(n).suffix.lower() for n in files_in_zip))
print('Example filenames:'); print('\n'.join(files_in_zip[:12]))
print('Metadata files:',[n for n in files_in_zip if Path(n).suffix.lower()!='.jpg'])


In [ ]:
PATTERN=re.compile(r'^Ripeness Grading/day (\d+)/([^/]+)/APPLE (\d+)/apple(\d+)[_-](\d+)\.jpg$',re.I)
rows=[]; unmatched=[]
for name in files_in_zip:
    m=PATTERN.fullmatch(name)
    if not m: unmatched.append(name); continue
    day,variety,apple,file_apple,capture=m.groups()
    assert apple==file_apple, f'Conflicting apple numbers: {name}'
    rows.append(dict(member=name,day=int(day),variety=variety.lower(),apple_number=int(apple),
                     candidate_specimen=f'{variety.lower()}/apple_{apple}',capture_index=int(capture),
                     wavelength_nm=None,stage=None))
inventory=pd.DataFrame(rows)
inventory.to_csv(OUT/'spectral_inventory_UNVERIFIED.csv',index=False)
display(inventory.head(12))
display(inventory.groupby('day').size().rename('image_count').to_frame())
print('Candidate variety + apple IDs:',inventory.candidate_specimen.nunique())
print('Capture indices (NOT verified wavelength bands):',sorted(inventory.capture_index.unique()))
print('Unparsed members:',unmatched[:20])
counts=pd.DataFrame({'stage':['under-ripe','ripe','over-ripe','spoiled'],
                     'verified_identifiable_physical_apples':['unknown','unknown','unknown','not supplied'],
                     'reason':['No per-observation stage key']*3+['Not a dataset class']})
display(counts); counts.to_csv(OUT/'physical_apples_per_class.csv',index=False)
issues=[
    'Need author-confirmed concatenated panel -> wavelength (nm), bandwidth, black-reference and filename suffix/view mapping.',
    'Need stage label for each specimen/day/view; days are not stage ground truth.',
    'Need confirmation that variety + apple number denotes the same physical apple across all days and views.',
    'Need acquisition/exposure and preprocessing details; JPEG intensity is not calibrated reflectance.'
]
save_json('spectral_status.json',dict(status='BLOCKED_UNVERIFIED_MAPPING',accuracy=None,
          candidate_specimens=int(inventory.candidate_specimen.nunique()),issues=issues))
print('NO SPECTRAL ACCURACY REPORTED.'); print('\n'.join(issues))


In [ ]:
def read_archive_members(names):
    """Read selected members only; cache bytes under a hash, never extract untrusted paths."""
    cached={}; missing=[]
    cache=WORK/'spectral_samples'; cache.mkdir(exist_ok=True)
    for name in names:
        p=cache/(hashlib.sha256(name.encode()).hexdigest()+'.jpg')
        if p.exists(): cached[name]=p.read_bytes()
        else: missing.append((name,p))
    if missing:
        archive=zipfile.ZipFile(LOCAL_ARCHIVE) if LOCAL_ARCHIVE else RemoteZip(archive_url,timeout=120)
        with archive as z:
            for name,p in missing:
                b=z.read(name); p.write_bytes(b); cached[name]=b
    return cached

sample_names=inventory[(inventory.day==1)&(inventory.variety=='kinnaur')&(inventory.apple_number==1)].sort_values('capture_index').member.tolist()
samples=read_archive_members(sample_names)
sample_info=[]
for name in sample_names:
    im=Image.open(io.BytesIO(samples[name]))
    sample_info.append(dict(member=name,width=im.width,height=im.height,mode=im.mode))
display(pd.DataFrame(sample_info))
pd.DataFrame(sample_info).to_csv(OUT/'spectral_sample_dimensions.csv',index=False)
im=Image.open(io.BytesIO(samples[sample_names[0]]))
print('Full strip:',sample_names[0],im.size)
display(im.resize((1440,90)))
# Diagnostic equal-width slicing only. Never use these positions as a band key.
if im.size == (17280,1080):
    fig,axes=plt.subplots(3,3,figsize=(12,8))
    for i,ax in enumerate(axes.flat):
        ax.imshow(im.crop((1920*i,0,1920*(i+1),1080)))
        ax.set_title(f'Panel {i+1}: wavelength unknown'); ax.axis('off')
    plt.tight_layout(); fig.savefig(OUT/'spectral_panel_contact_sheet.png'); plt.show()
else:
    print('Layout changed; inspect manually before slicing.')
print('Concatenation suggests within-file pairing, but wavelengths, stages and persistent specimen identity remain unverified.')


### Conditional training path — only after the missing mappings are obtained
The current public archive cannot support a defensible spectral classifier by itself. This default run finishes with an audit, **not a fabricated accuracy or model**. Do not sort unrelated files and zip them into spectral rows, label early/late days by assumption, or split individual images randomly.

The cell below is a future, explicitly gated training implementation. An author-verified manifest must contain one row per verified wavelength panel with `member, specimen_id, observation_id, stage, wavelength_nm, evidence_url, x1, y1, x2, y2` (panel bounds in its source JPEG). An observation is one specimen at one time and view. Specimen IDs must persist across days, stages and views. Exclude black/reference images from the feature rows after documenting their identity. Supply the verified band order and provenance JSON as well. Neither file is required for today's detection/audit run.

Feature: median grayscale JPEG intensity inside a manually reviewed fruit mask for each band (no whole-image mean). These are **camera/illumination features**, not calibrated AS7341 measurements. Masks must be prepared independently of labels; save them as 0/255 PNGs with the cropped panel dimensions. The mask manifest supplies `mask_path` for each row. Manual review is required because the fruit sits among background/chamber objects and an automatic threshold is not established here.


In [ ]:
# Leave unset today. Later place author-verified manifest/provenance and reviewed masks in Drive.
VERIFIED_MANIFEST = None
VERIFIED_PROVENANCE = None

def train_verified_spectral(manifest_path, provenance_path):
    from sklearn.model_selection import GroupShuffleSplit
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score, accuracy_score
    import joblib
    provenance=json.loads(Path(provenance_path).read_text())
    required_evidence=['identity_across_days_and_views','stage_mapping','panel_to_wavelength_mapping','mask_review']
    assert all(provenance.get(k) for k in required_evidence), 'Missing evidence references'
    bands=[float(x) for x in provenance['band_order_nm']]
    assert len(bands)==8 and len(set(bands))==8, 'Verify all eight bands and exclude reference captures'
    m=pd.read_csv(manifest_path)
    required=['member','specimen_id','observation_id','stage','wavelength_nm','evidence_url','mask_path','x1','y1','x2','y2']
    assert set(required)<=set(m.columns)
    assert not m[required].isna().any().any()
    assert set(m.member)<=set(files_in_zip)
    assert not m.duplicated(['member','x1','y1','x2','y2']).any(), 'Repeated panel in manifest'
    assert set(m.stage)<= {'under-ripe','ripe','over-ripe'}, 'Do not relabel over-ripe as spoiled'
    assert not m.duplicated(['observation_id','wavelength_nm']).any()
    for _,g in m.groupby('observation_id'):
        assert g.specimen_id.nunique()==g.stage.nunique()==1
        assert set(g.wavelength_nm.astype(float))==set(bands), 'Incomplete/mismatched observation'
    # Keep all repeated days and views of a specimen together, even if its stage changes.
    blobs=read_archive_members(m.member.tolist()); feature_rows=[]
    for obs,g in m.groupby('observation_id',sort=True):
        row=dict(observation_id=obs,specimen_id=g.specimen_id.iloc[0],stage=g.stage.iloc[0])
        for r in g.itertuples():
            im=Image.open(io.BytesIO(blobs[r.member])).convert('L')
            bounds=[r.x1,r.y1,r.x2,r.y2]
            assert all(float(v).is_integer() for v in bounds)
            x1,y1,x2,y2=map(int,bounds)
            assert 0 <= x1 < x2 <= im.width and 0 <= y1 < y2 <= im.height
            a=np.asarray(im.crop((x1,y1,x2,y2)),dtype=float)
            mask=np.asarray(Image.open(r.mask_path).convert('L'))>127
            assert mask.shape==a.shape and 0.01 < mask.mean() < 0.95, 'Invalid fruit mask'
            row[f'nm_{float(r.wavelength_nm):g}']=float(np.median(a[mask])/255.)
        feature_rows.append(row)
    features=pd.DataFrame(feature_rows); columns=[f'nm_{b:g}' for b in bands]
    assert np.isfinite(features[columns].to_numpy()).all()
    # One prespecified split. Never search random seeds for a better score.
    train_i,val_i=next(GroupShuffleSplit(n_splits=1,test_size=0.25,random_state=SEED).split(features,groups=features.specimen_id))
    tr,va=features.iloc[train_i],features.iloc[val_i]
    assert set(tr.specimen_id).isdisjoint(va.specimen_id)
    assert set(tr.stage)==set(va.stage)==set(features.stage), 'Too few independent specimens for this split; collect more data'
    features['split']='train'; features.loc[val_i,'split']='validation'
    features.to_csv(OUT/'spectral_features_and_split.csv',index=False)
    display(features.groupby(['split','stage']).specimen_id.nunique().rename('physical_apples').to_frame())
    model=make_pipeline(StandardScaler(),LogisticRegression(class_weight='balanced',max_iter=2000,random_state=SEED))
    model.fit(tr[columns],tr.stage); pred=model.predict(va[columns])
    labels=list(model.classes_)
    metrics=dict(accuracy=accuracy_score(va.stage,pred),balanced_accuracy=balanced_accuracy_score(va.stage,pred),
                 report=classification_report(va.stage,pred,output_dict=True,zero_division=0),
                 confusion_matrix=confusion_matrix(va.stage,pred,labels=labels).tolist(),labels=labels,
                 validation_specimens=va.specimen_id.nunique(),validation_observations=len(va),
                 limitation='Observation-level metrics on a grouped small holdout; correlated days/views are not independent test specimens.')
    save_json('spectral_metrics.json',metrics)
    save_json('spectral_provenance.json',provenance)
    save_json('spectral_schema.json',dict(features=columns,labels=labels,feature='fruit-mask median grayscale JPEG / 255',
                                        sensor='camera under controlled illumination; NOT AS7341',spoiled_supported=False))
    shutil.copy2(manifest_path,OUT/'verified_spectral_manifest.csv')
    joblib.dump(model,OUT/'spectral_classifier.joblib')
    save_json('spectral_status.json',dict(status='TRAINED_WITH_SUPPLIED_VERIFIED_METADATA',metrics_file='spectral_metrics.json',as7341_ready=False,spoiled_supported=False))
    display(pd.DataFrame(metrics['report']).T)
    return model

if VERIFIED_MANIFEST is not None and VERIFIED_PROVENANCE is not None:
    spectral_model=train_verified_spectral(VERIFIED_MANIFEST,VERIFIED_PROVENANCE)
else:
    print('Spectral training correctly skipped: unverified specimen, stage and wavelength mappings. Accuracy: not available.')


## Demo handoff and run limits

| Output | Runs now | Later input |
|---|---|---|
| YOLO baseline, numbered boxes, PNG crops + CSV | Public fruit image, CPU or GPU | Phone/gantry RGB photo via `detect_and_export(path, tag='live_photo')`; EXIF orientation corrected |
| Optional fine-tuned detector + comparison | Public COCO subset, preferably T4 | Same RGB path; compare before adopting |
| Spectral inventory + contact sheet + missing-mapping report | Public Mendeley archive, no hardware | Author-confirmed capture/stage/specimen metadata |
| Spectral classifier | Blocked with current public metadata | Verified paired band images + reviewed masks; grouped holdout |
| Sujal's RGB ripeness predictions | Requires his separately trained ONNX + label file | Feed exported crops using his exact preprocessing |

Optional phone upload (run manually):
```python
from google.colab import files
uploaded = files.upload()
for i, name in enumerate(uploaded):
    detect_and_export(name, tag=f'phone_{i}')
```

**AS7341 gate:** Its eight visible channels, NIR and clear channels measure a different instrument response from webcam JPEGs under switched illumination. Confirm actual channel response curves/bandwidths and ordering from the [manufacturer documentation](https://ams-osram.com/products/sensors/ambient-light-color-spectral-proximity-sensors/ams-as7341-11-channel-spectral-color-sensor). Match illumination geometry, distance, gain/integration time, dark/white references, saturation handling and normalization. Matching nominal wavelengths alone is insufficient. Obtain paired camera/AS7341 measurements on additional, independently labeled apples, fit/validate a transfer calibration or retrain a sensor-specific model, and test on held-out physical apples. Do not feed raw AS7341 counts into this camera model. An ordinary phone photo is also not an eight-band observation.

Reserve all three physical apples for the live demonstration. The rotten apple is a **spoiled** example, not an `over-ripe` training label. Mendeley supplies no verified spoiled class, so a spectral model trained on it cannot certify spoilage or food safety; display unsupported/unknown for that task.

**Storage:** Large data and caches stay under `/content`; Drive stores models, small results, provenance and crop outputs. Download/cache failures are surfaced, not silently turned into empty training data. Temporary Colab data disappears on runtime reset. Each new run has its own Drive directory.

Sources: [Ultralytics detection API](https://docs.ultralytics.com/tasks/detect/), [COCO](https://cocodataset.org/#download) (retain image-specific licenses in exported metadata), [COCO demo image source on Flickr](https://www.flickr.com/photo.gne?id=4430551771) (COCO license ID 4: CC BY 2.0; see original image attribution), [Mendeley v2, Gaikwad & Kothari](https://doi.org/10.17632/y5h6v8w6ms.2) (CC BY 4.0; derived inventories/contact sheets are our modifications). Check source/software licensing for redistribution.

**Testing:** See repository `docs/detection_spectral_validation.md` for actual development checks. Colab Drive authorization and GPU fine-tuning must be run in your own Colab session. No spectral accuracy is claimed.


In [ ]:
print('Run complete. Drive artifacts:')
for p in sorted(OUT.rglob('*')):
    if p.is_file(): print(p.relative_to(OUT))
